# Chapter 4 — Semantic Search from Scratch

This notebook accompanies **Chapter 4** of *Build an Advanced RAG Application (From Scratch)*.

We start with a corpus of hotel reviews and build a semantic search engine in three increasingly fast forms:

1. **Pure NumPy** — cosine similarity by hand.
2. **NumPy with normalized Euclidean distance** — same ranking, different metric.
3. **FAISS** — the production-grade vector search library.

We finish by comparing FAISS index types (Flat / HNSW / IVF-PQ) on the same query.

> Reusable code lives in `data_loader.py` and `search.py` next to this notebook.


## 1. Setup

Make sure you're in the `advanced-rag` conda env (see the root README) and that the kernel for this notebook points to it.

In [2]:
import os, sys, time
import numpy as np
import torch

# Make the chapter folder importable
sys.path.insert(0, os.path.dirname(os.path.abspath('.')) if not os.path.exists('search.py') else '.')

from data_loader import load_paris_reviews
from search import (
    load_embedding_model,
    get_embeddings,
    cosine_search,
    euclidean_search,
    build_faiss_cosine_index,
    search_faiss_index,
    build_faiss_indices,
)


## 2. Load the Paris hotel reviews

The dataset is hosted on the HuggingFace Hub. The first call downloads it; subsequent calls hit the local cache.

In [3]:
df_paris = load_paris_reviews()
print(f"Rows: {len(df_paris):,}")
df_paris.head()

README.md:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

c:\Users\Areej\Desktop\traversaal\advanced-rag-from-scratch\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Areej\.cache\huggingface\hub\datasets--traversaal-ai-hackathon--hotel_datasets. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


hotel_reviews_Istanbul.csv: 0.00B [00:00, ?B/s]

hotel_reviews_San%20Francisco.csv: 0.00B [00:00, ?B/s]

hotel_reviews_london.csv: 0.00B [00:00, ?B/s]

hotel_reviews_nyc.csv: 0.00B [00:00, ?B/s]

hotel_reviews_paris.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5997 [00:00<?, ? examples/s]

Rows: 1,200


,hotel_name,hotel_description,review_title,review_text,rate,tripdate,hotel_url,hotel_image,price_range,rating_value,review_count,street_address,locality,country
0,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,Awesome Paris Hotel!,"Fantastic hotel! Awesome location, great chara...",5.0,January 2024,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France
1,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,Charming Hotel,Charming Hotel in a central location. The sta...,5.0,May 2023,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France
2,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,Highly recommend this hotel,Highly recommend this hotel and we would absol...,5.0,December 2023,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France
3,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,"Good location, excellent staff and large room",Good central location - close to Metro and man...,5.0,December 2023,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France
4,Hotel Malte - Astotel,Located in the 2nd district next to the Stock ...,"Good staff, quiet and beautiful location.","Lovely staff. All were good, and Manon was out...",5.0,January 2024,https://www.tripadvisor.com/Hotel_Review-g1871...,https://media-cdn.tripadvisor.com/media/photo-...,$$ (Based on Average Nightly Rates for a Stand...,5.0,2985,63 rue de Richelieu,Paris,France


In [4]:
df_paris.hotel_name.value_counts().head(10)

hotel_name
Hotel Malte - Astotel           40
Hotel Astoria - Astotel         40
Novotel Paris Les Halles        40
La Maison Favart                40
Grand Hotel du Palais Royal     40
Hotel Maison Mere               40
Hotel des Arts - Montmartre     40
Hotel Joke - Astotel            40
Passy Eiffel Hotel              40
Best Western Plus La Demeure    40
Name: count, dtype: int64

## 3. Embed the reviews

We use `nomic-ai/nomic-embed-text-v1.5` — a strong open-weight 768-dim embedding model. On CPU this takes a few minutes; on GPU it's much faster.

In [5]:
model = load_embedding_model()

if torch.cuda.is_available():
    model = model.to("cuda")
    print("CUDA available — model on GPU.")
elif torch.backends.mps.is_available():
    model = model.to("mps")
    print("MPS available — model on Apple Silicon GPU.")
else:
    print("Running on CPU.")

modules.json:   0%|          | 0.00/255 [00:00<?, ?B/s]

c:\Users\Areej\Desktop\traversaal\advanced-rag-from-scratch\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Areej\.cache\huggingface\hub\models--nomic-ai--nomic-embed-text-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/140 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

<All keys matched successfully>


config.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

Running on CPU.


In [6]:
reviews = df_paris["review_text"].tolist()
review_embeddings = model.encode(reviews, show_progress_bar=True).astype("float32")
print(f"Embeddings shape: {review_embeddings.shape}")

Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Embeddings shape: (1200, 768)


## 4. Search by hand: cosine similarity

No libraries needed. We compute dot products of L2-normalized vectors.

In [7]:
query = "Hotel with a view of the Eiffel tower."
query_embedding = model.encode([query]).astype("float32")

t0 = time.time()
indices, sims = cosine_search(query_embedding, review_embeddings, k=5)
print(f"Cosine search took {time.time()-t0:.4f}s")

print(f"\nQuery: {query}\n")
for rank, (idx, sim) in enumerate(zip(indices, sims), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (sim={sim:.4f})")
    print(f"   {df_paris.iloc[idx]['review_text'][:200]}...\n")

Cosine search took 0.0113s

Query: Hotel with a view of the Eiffel tower.

1. Pullman Paris Eiffel Tower Hotel  (sim=0.8227)
   Excellent service. Stunning view of the Eiffel towere from our balcony, The room is gorgeous, comfortable and spacious. Definitely will be recommending this hotel to family and friends. If you’re look...

2. Pullman Paris Eiffel Tower Hotel  (sim=0.8078)
   If you stay at this hotel it is for the amazing views of the Eiffel Tower and for the pictures.  The photos and memories of being on the balcony looking at the Eiffel Tower in all its splendor cannot ...

3. Citadines Tour Eiffel Paris  (sim=0.7935)
   Had a Eiffel tower balcony view room and the view did not disappoint was absolutely amazing and so nice to see everyone from the room or balcony. Staff at reception were nice and was no problem in che...

4. Hotel Marignan Champs-Elysees  (sim=0.7929)
   Very good hotel near the Champs-Elysees.  The superior rooms are very nice and a good size.  The Eiffel To

## 5. Same ranking via normalized Euclidean distance

For unit-norm vectors, `||a − b||² = 2(1 − cos(a, b))`, so Euclidean distance ranks identically to cosine similarity (just inverted).

In [8]:
indices, distances = euclidean_search(query_embedding, review_embeddings, k=5)
for rank, (idx, dist) in enumerate(zip(indices, distances), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (dist={dist:.4f})")

1. Pullman Paris Eiffel Tower Hotel  (dist=0.5955)
2. Pullman Paris Eiffel Tower Hotel  (dist=0.6200)
3. Citadines Tour Eiffel Paris  (dist=0.6426)
4. Hotel Marignan Champs-Elysees  (dist=0.6435)
5. Hotel Tourisme Avenue  (dist=0.6459)


## 6. Scale up with FAISS

NumPy is fine for a few thousand rows but quickly falls over. FAISS is purpose-built for vector search.

We build an `IndexFlatIP` (inner product) on **L2-normalized** vectors — that's mathematically equivalent to exact cosine similarity but with FAISS's optimized SIMD search.

In [9]:
faiss_index = build_faiss_cosine_index(review_embeddings)

t0 = time.time()
distances, indices = search_faiss_index(query_embedding, faiss_index, k=5)
print(f"FAISS search took {time.time()-t0:.4f}s\n")

for rank, (idx, sim) in enumerate(zip(indices[0], distances[0]), 1):
    print(f"{rank}. {df_paris.iloc[idx]['hotel_name']}  (cos={sim:.4f})")

FAISS search took 0.0045s

1. Pullman Paris Eiffel Tower Hotel  (cos=0.8227)
2. Pullman Paris Eiffel Tower Hotel  (cos=0.8078)
3. Citadines Tour Eiffel Paris  (cos=0.7935)
4. Hotel Marignan Champs-Elysees  (cos=0.7929)
5. Hotel Tourisme Avenue  (cos=0.7914)


### Aggregate by hotel

A single hotel may show up many times in the top-k. Group by hotel and rank by mean cosine to surface *places* rather than individual reviews.

In [10]:
k = 120
distances, indices = search_faiss_index(query_embedding, faiss_index, k=k)

hotels = {}
for idx, dist in zip(indices[0], distances[0]):
    name = df_paris.iloc[idx]["hotel_name"]
    h = hotels.setdefault(name, {"reviews": [], "scores": []})
    h["reviews"].append(df_paris.iloc[idx]["review_text"])
    h["scores"].append(float(dist))

ranked = sorted(
    [(n, np.mean(h["scores"]), len(h["reviews"])) for n, h in hotels.items() if len(h["reviews"]) >= 2],
    key=lambda t: t[1],
    reverse=True,
)
for name, mean_score, n in ranked[:10]:
    print(f"{mean_score:.4f}  {n:3d} reviews  {name}")

0.7461   10 reviews  Citadines Tour Eiffel Paris
0.7413   16 reviews  Pullman Paris Eiffel Tower Hotel
0.7394   18 reviews  Passy Eiffel Hotel
0.7377    5 reviews  citizenM Paris Champs-Elysees
0.7377    6 reviews  Hotel Marignan Champs-Elysees
0.7357   12 reviews  Hotel Tourisme Avenue
0.7350   19 reviews  Hotel La Comtesse
0.7244   11 reviews  Cler Hotel
0.7225    3 reviews  Best Western Plus La Demeure
0.7207    2 reviews  Hotel Campanile Paris Bercy Village


## 7. Comparing FAISS index types

- **Flat** — exact, full-scan; slow on big corpora.
- **HNSW** — graph-based, fast & accurate, more memory.
- **IVF-PQ** — clustered + quantized, very fast, lossy.

For a small corpus the latency differences are tiny; on millions of vectors they're decisive.

In [11]:
indices_set = build_faiss_indices(review_embeddings)
k = 15

for name, idx in indices_set.items():
    t0 = time.time()
    distances, top = idx.search(query_embedding, k)
    print(f"{name:6s}: {time.time()-t0:.4f}s  unique hotels = {len({df_paris.iloc[i]['hotel_name'] for i in top[0]})}")

flat  : 0.0005s  unique hotels = 7
hnsw  : 0.0009s  unique hotels = 7
ivfpq : 0.0016s  unique hotels = 7


## What's next

Chapter 5 plugs the **decoder** (LLM) on top of these retrievals to start producing grounded answers — the first half of a full RAG pipeline. Chapter 6 wires retrieval + generation together end-to-end and adds a real vector database (Qdrant).